In [ ]:
"""
LangGraph ReAct agent + MySQL tools (schema, NL->SQL, execute)
Compatible with:
  - langchain==1.0.3
  - langchain-community==0.4.1
  - langchain-openai==1.0.1
  - langgraph>=0.2.x
  - duckduckgo-search>=6.x

Env you need:
  MYSQL_HOST, MYSQL_PORT, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB
  OPENAI_API_KEY
  (optional) MYSQL_MAX_ROWS
"""

from dotenv import load_dotenv
load_dotenv()

import os
import time
from typing import Any, Optional, List
import re

# --- LangChain / LangGraph imports (v1-compatible) ---------------------------
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import BaseTool
from langchain_classic.chains import create_sql_query_chain  # v1.x re-export
from langgraph.prebuilt import create_react_agent

# --- SQLAlchemy for safe execution ------------------------------------------
from sqlalchemy import create_engine, text


# ====================== Global agent system prompt ===========================

AGENT_SYSTEM_PROMPT = """
You are a helpful senior data assistant that talks to a MySQL database using tools.

Your main goals:
1. Understand the user's business question.
2. If the question does NOT contain enough specific information to build a precise SQL query,
   you MUST ask the user follow-up questions BEFORE calling any MySQL tools.

Examples of missing information you must clarify:
- Which table(s) to use, if it is ambiguous.
- Which experiment_id, user_id, or other identifiers to filter on.
- Required date range or time window (e.g. "last 7 days", "2024 only").
- Which metric(s) or columns to aggregate, and how (SUM / AVG / MAX / MIN / COUNT).
- Any grouping needed (e.g. "by experiment", "by day", "by dataset").
- Any filters for environment, status, tags, etc.

Guidelines:
- Do NOT guess table or column names.
- Do NOT assume default filters like experiment_id = 1 unless the user said so.
- When clarification is needed, ask the user directly in natural language.
- Ask for all obviously needed details in as few follow-up questions as possible.
- Only after you have the necessary details, use the MySQL tools to:
  (1) inspect schema if needed,
  (2) generate SQL from natural language,
  (3) execute the SQL and return the results.

If the user query is already specific enough to write an unambiguous SELECT query,
you can directly use the MySQL tools.
"""


# ====================== Helpers =============================================

def _mysql_uri_from_env() -> str:
    host = os.getenv("MYSQL_HOST", "localhost")
    port = int(os.getenv("MYSQL_PORT", "3306"))
    user = os.getenv("MYSQL_USER")
    pwd = os.getenv("MYSQL_PASSWORD")
    db = os.getenv("MYSQL_DB")
    if not all([user, pwd, db]):
        raise RuntimeError("Set MYSQL_USER, MYSQL_PASSWORD, and MYSQL_DB in env")
    return f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}"


def _get_db() -> SQLDatabase:
    # Use SQLDatabase for schema & dialect awareness
    return SQLDatabase.from_uri(_mysql_uri_from_env())


def _get_engine():
    # Execute with SQLAlchemy engine (avoids relying on SQLDatabase internals)
    return create_engine(_mysql_uri_from_env(), pool_pre_ping=True)


_SELECT_ONLY = re.compile(r"(?is)^\s*select\b")


def _is_select_only(sql: str) -> bool:
    s = sql.strip()
    if ";" in s:
        return False
    return bool(_SELECT_ONLY.match(s))


# ====================== Tool 1: Schema discovery ============================

class MySQLSchemaTool(BaseTool):
    name: str = "mysql_schema_tool"
    description: str = (
        "Discover MySQL schema. "
        "Input: empty (lists all tables) OR a comma-separated list of tables. "
        "Use this to understand which tables/columns exist before building SQL."
    )

    def _run(self, table_list: str = "") -> str:
        db = _get_db()
        engine = _get_engine()
        with engine.connect() as conn:
            if not table_list.strip():
                tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]
            else:
                tables = [t.strip() for t in table_list.split(",") if t.strip()]
            if not tables:
                return "No tables found."

            lines: List[str] = []
            for t in tables:
                try:
                    rows = conn.execute(text(f"DESCRIBE `{t}`")).fetchall()
                except Exception as e:
                    lines.append(f"# {t}\nERROR: {e}")
                    continue
                lines.append(f"# {t}")
                for field, col_type, nullable, key, default, extra in rows:
                    lines.append(
                        f"- {field}: {col_type}, NULL={nullable}, KEY={key}, "
                        f"DEFAULT={default}, EXTRA={extra}"
                    )
        # Append a compact database-level table info snapshot (first few lines)
        try:
            snapshot = db.get_table_info()
            if snapshot:
                head = "\n".join(snapshot.splitlines()[:20])
                lines.append("\n# table_info snapshot (truncated):\n" + head)
        except Exception:
            pass
        return "\n".join(lines)

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Tool 2: Natural language -> SQL =====================

class MySQLNL2SQLTool(BaseTool):
    name: str = "mysql_nl2sql_tool"
    description: str = (
        "Generate a single, safe MySQL SELECT query from a natural-language question. "
        "Uses the database schema loaded from the local file 'mlflow_db.txt'. "
        "Use this ONLY AFTER the user has provided all necessary details "
        "(tables, filters like experiment_id, date ranges, metrics, grouping, etc.). "
        "Input: the user question. Output: SQL only, one SELECT statement, no semicolons."
    )
    llm: Any = None  # ChatOpenAI injected at construction

    def _run(self, question: str) -> str:
        if not question or not question.strip():
            return "Please provide a question."

        # ---- Load schema from file -----------------------------------------
        try:
            with open("mlflow_db.txt", "r", encoding="utf-8") as f:
                schema_text = f.read()
        except Exception as e:
            return f"Could not read schema file 'mlflow_db.txt': {e}"

        # ---- Prompt LLM with schema + question -----------------------------
        system = (
            "You are a senior data analyst. Given a MySQL database schema and a "
            "natural language question, produce ONE valid MySQL SELECT query that "
            "answers the question using only the tables and columns described in the "
            "schema. No comments, no explanations, no DDL/DML, SELECT-only."
            "- Always wrap table names and column names in backticks (`) if they contain spaces, special characters, or match MySQL reserved keywords.\n"
        )

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system),
                (
                    "human",
                    "Database schema (from file mlflow_db.txt):\n"
                    "{schema}\n\n"
                    "Question:\n"
                    "{question}\n\n"
                    "Constraints:\n"
                    "- Only one SELECT statement\n"
                    "- No semicolons\n"
                    "- MySQL dialect\n\n"
                    "-- Always wrap table names and column names in backticks (`) if they contain spaces, special characters, or match MySQL reserved keywords.\n"
                    "SQL:"
                ),
            ]
        )

        messages = prompt.format_messages(
            schema=schema_text,
            question=question.strip(),
        )

        response = self.llm.invoke(messages)
        sql = getattr(response, "content", str(response)).strip()

        # ---- Safety: enforce a single SELECT without semicolons ------------
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        if not _is_select_only(sql):
            return "Failed to produce a safe single SELECT statement. Try rephrasing."

        return sql

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Tool 3: Execute SQL (read-only) =====================

class MySQLQueryExecTool(BaseTool):
    name: str = "mysql_query_exec_tool"
    description: str = (
        "Execute a single SELECT query against MySQL and return rows (capped). "
        "Input must be ONE SELECT statement, no semicolons. "
        "Use this ONLY AFTER the SQL has been clearly defined/validated by the conversation. "
        "Do NOT guess or generate SQL inside this tool."
    )

    def _run(self, sql: str) -> str:
        if not _is_select_only(sql):
            return "Only a single SELECT statement without ';' is allowed."
        row_cap = int(os.getenv("MYSQL_MAX_ROWS", "200"))
        engine = _get_engine()
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            rows = result.fetchmany(row_cap + 1)
            headers = list(result.keys())
        truncated = len(rows) > row_cap
        rows = rows[:row_cap]

        out = []
        header_line = " | ".join(map(str, headers))
        out.append(header_line)
        out.append("-" * max(3, len(header_line)))
        for r in rows:
            out.append(" | ".join("" if v is None else str(v) for v in r))
        if truncated:
            out.append(f"...(truncated at {row_cap} rows)")
        return "\n".join(out)

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Agent builder =======================================

def build_web_agent(model_name: Optional[str] = None):
    """Create a LangGraph ReAct agent with DuckDuckGo + MySQL tools."""
    llm_id = model_name or os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    llm = ChatOpenAI(model=llm_id, temperature=0)

    ddg = DuckDuckGoSearchRun()
    mysql_schema = MySQLSchemaTool()
    mysql_nl2sql = MySQLNL2SQLTool(llm=llm)
    mysql_exec = MySQLQueryExecTool()

    tools = [ddg, mysql_schema, mysql_nl2sql, mysql_exec]

    # Inject the system prompt so the agent always asks for missing info
    agent = create_react_agent(
        llm,
        tools=tools,
        prompt=AGENT_SYSTEM_PROMPT,
    )
    return agent


# ====================== Interactive CLI demo ================================

if __name__ == "__main__":
    agent = build_web_agent()
    print("[demo] Interactive MySQL assistant is running.")
    print("Type your question (or 'exit' / 'quit' to stop).\n")

    # Conversation history for the ReAct agent
    messages: List[tuple[str, str]] = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[demo] Exiting.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("[demo] Exiting.")
            break

        # Append the new user message
        messages.append(("user", user_input))

        start = time.time()
        # Call the agent with full conversation so far
        result = agent.invoke({"messages": messages})
        elapsed = time.time() - start

        # result should be a dict with "messages"
        msgs = result.get("messages") if isinstance(result, dict) else result
        last = msgs[-1]
        content = getattr(last, "content", str(last))

        print(f"\nAssistant: {content}\n")
        print(f"[demo] Turn completed in {elapsed:.2f}s\n")

        # Add assistant message to history for context in the next turn
        messages.append(("assistant", content))


[demo] Interactive MySQL assistant is running.
Type your question (or 'exit' / 'quit' to stop).



C:\Users\surya.adatravu\AppData\Local\Temp\ipykernel_28548\1838081271.py:274: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


You:  Which registered models have aliases pointing to versions that are currently archived?



Assistant: To provide you with the correct information, I need to clarify a few details:

1. Which table(s) should I look at to find the registered models and their aliases?
2. Are there any specific filters you want to apply, such as a particular model name or a date range?
3. Do you want any specific columns in the result, or is it sufficient to list the model names and their aliases?

Once I have this information, I can help you with the query.

[demo] Turn completed in 2.65s



You:  models_version, registered_models,


...


Assistant: I have retrieved the schema for the relevant tables. Here are the key tables and their columns:

1. **registered_models**
   - `name`: The name of the registered model.
   - `creation_time`: The time the model was created.
   - `last_updated_time`: The time the model was last updated.
   - `description`: A description of the model.

2. **model_versions**
   - `name`: The name of the model.
   - `version`: The version number.
   - `status`: The status of the version (this may indicate if it's archived).
   - `current_stage`: The lifecycle stage of the version.

3. **registered_model_aliases**
   - `alias`: The alias for the registered model.
   - `version`: The version number associated with the alias.
   - `name`: The name of the registered model.

To find all registered models that have aliases pointing to versions that are currently archived, I will need to filter based on the `status` column in the `model_versions` table.

Could you please confirm that the `status` colum

You:  all columns



Assistant: Here are the relevant columns from the tables:

1. **model_versions**
   - `name`: The name of the model.
   - `version`: The version number.
   - `status`: The status of the version (this will help us identify archived versions).
   - `current_stage`: The lifecycle stage of the version.
   - Other columns include `creation_time`, `last_updated_time`, `description`, etc.

2. **registered_model_aliases**
   - `alias`: The alias for the registered model.
   - `version`: The version number associated with the alias.
   - `name`: The name of the registered model.

To find all registered models that have aliases pointing to versions that are currently archived, I will assume that the `status` column in the `model_versions` table indicates whether a version is archived. 

Could you please confirm what value in the `status` column indicates that a version is archived?

[demo] Turn completed in 6.19s



You:  status is archive



Assistant: The query has been executed successfully, but it appears that there are no results returned. This means that there are currently no registered models with aliases pointing to versions that are archived in the database.

If you have any other questions or need further assistance, feel free to ask!

[demo] Turn completed in 15.52s

